# YOLOv8n Training — 40 Food Classes
Pipeline:
1. Mount Google Drive and copy dataset to local disk
2. Auto-annotate: generate YOLO bbox labels using Otsu edge detection
3. Build YOLO dataset structure
4. Train YOLOv8n (pretrained COCO weights)
5. Validate and download best.pt

Place `food_dataset/` folder in Google Drive root before running.

In [1]:
# ── 1. Install ────────────────────────────────────────────────────────────
!pip install -q ultralytics opencv-python-headless

import os, zipfile, shutil, cv2, yaml
import numpy as np
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

print('Setup complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete


In [2]:
# ── 2. Upload & extract dataset ───────────────────────────────────────────
from google.colab import files
print('Upload food_dataset.zip')
uploaded = files.upload()   # upload food_dataset.zip

with zipfile.ZipFile('food_dataset.zip', 'r') as z:
    z.extractall('.')

DATASET_PATH = 'food_dataset'
print('Train classes:', len(os.listdir(f'{DATASET_PATH}/train')))

Upload food_dataset.zip


Saving food_dataset.zip to food_dataset.zip
Train classes: 40


In [5]:
# ── 3. Auto-annotate ──────────────────────────────────────────────────────
# Single food per image -> tight bbox via Otsu edge detection
# Falls back to full-image bbox if contour is too small

def get_tight_bbox(img_path):
    """
    Returns normalized YOLO bbox (cx, cy, w, h).
    Uses Otsu threshold + largest contour for tight bbox.
    Falls back to full-image (0.5, 0.5, 1.0, 1.0) if detection fails.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return 0.5, 0.5, 1.0, 1.0

    h, w = img.shape[:2]

    try:
        gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return 0.5, 0.5, 1.0, 1.0

        c = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(c)

        # 5% padding
        pad_x = int(bw * 0.05)
        pad_y = int(bh * 0.05)
        x  = max(0, x - pad_x)
        y  = max(0, y - pad_y)
        bw = min(w - x, bw + 2 * pad_x)
        bh = min(h - y, bh + 2 * pad_y)

        # Reject if bbox covers less than 10% of image
        if (bw * bh) < (w * h * 0.10):
            return 0.5, 0.5, 1.0, 1.0

        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        nw = bw / w
        nh = bh / h
        return round(cx, 6), round(cy, 6), round(nw, 6), round(nh, 6)

    except Exception:
        return 0.5, 0.5, 1.0, 1.0


def annotate_split(split):
    split_dir = Path(DATASET_PATH) / split
    total, tight, fallback = 0, 0, 0

    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_id = CLASS_TO_ID[cls_dir.name]

        for img_path in cls_dir.glob('*'):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.webp'):
                continue
            cx, cy, nw, nh = get_tight_bbox(img_path)
            img_path.with_suffix('.txt').write_text(f'{cls_id} {cx} {cy} {nw} {nh}\n')
            total += 1
            if nw < 1.0 or nh < 1.0:
                tight += 1
            else:
                fallback += 1

    print(f'  [{split}] {total} labels | tight: {tight} | full-image fallback: {fallback}')


# Define CLASS_NAMES, NUM_CLASSES, and CLASS_TO_ID
CLASS_NAMES = sorted([d.name for d in (Path(DATASET_PATH) / 'train').iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print('Generating YOLO annotations...')
annotate_split('train')
annotate_split('val')
print('Done.')

Generating YOLO annotations...
  [train] 3034 labels | tight: 1285 | full-image fallback: 1749
  [val] 767 labels | tight: 321 | full-image fallback: 446
Done.


In [7]:
# ── 4. Build YOLO dataset structure ──────────────────────────────────────
YOLO_DIR = Path('yolo_dataset')

for split in ['train', 'val']:
    img_out = YOLO_DIR / 'images' / split
    lbl_out = YOLO_DIR / 'labels' / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for cls_dir in (Path(DATASET_PATH) / split).iterdir():
        if not cls_dir.is_dir():
            continue
        for img_path in cls_dir.glob('*'):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.webp'):
                continue
            lbl_path = img_path.with_suffix('.txt')
            # Prefix with class name to avoid filename collisions
            stem = f'{cls_dir.name}__{img_path.name}'
            shutil.copy2(img_path, img_out / stem)
            if lbl_path.exists():
                shutil.copy2(lbl_path, lbl_out / stem.replace(img_path.suffix, '.txt'))

    n_imgs = len(list(img_out.glob('*')))
    print(f'  [{split}] {n_imgs} images copied')

data_yaml = {
    'path':  str(YOLO_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    NUM_CLASSES,
    'names': CLASS_NAMES
}
with open(YOLO_DIR / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f'data.yaml written — {NUM_CLASSES} classes')

  [train] 3034 images copied
  [val] 767 images copied
data.yaml written — 40 classes


In [8]:
# ── 5. Train YOLOv8n ──────────────────────────────────────────────────────
model = YOLO('yolov8n.pt')   # pretrained COCO weights

results = model.train(
    data     = str(YOLO_DIR / 'data.yaml'),
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = 'food_yolov8n',
    patience = 15,
    augment  = True,
    degrees  = 10,
    flipud   = 0.1,
    fliplr   = 0.5,
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    mosaic   = 1.0,
    mixup    = 0.1,
    device   = 0,
    workers  = 2,
    verbose  = True,
)

print('Training complete')
print(f'Best mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=food_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, 

In [9]:
# ── 6. Validate ───────────────────────────────────────────────────────────
best_model = YOLO('runs/detect/food_yolov8n/weights/best.pt')
metrics = best_model.val(data=str(YOLO_DIR / 'data.yaml'))
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,013,448 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1849.0±488.9 MB/s, size: 47.6 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 767 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 767/767 292.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 5.0it/s 9.7s
                   all        767        767      0.791      0.736      0.802      0.687
               ariselu         20         20      0.766        0.8      0.852      0.793
               biryani         20         20      0.926          1      0.995      0.941
        butter_chicken         20         20      0.869       0.85      0.943      0.836
          caesar_salad         20         20      0.832      0.743      0.849      0.789
               chapati         20         20 

In [10]:
# ── 7. Download ───────────────────────────────────────────────────────────
from google.colab import files
files.download('runs/detect/food_yolov8n/weights/best.pt')
print('Rename to food_yolov8.pt and place in backend/models/')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Rename to food_yolov8.pt and place in backend/models/
